### **PHY 321 Semester Project** ###

**Names of collaborators:**
- Matrim Cirullo-Nesbitt
- Abhishek Vilekar
- Andrew Gabaldon


### **How Does Other Forces Influence a Golf Ball's Trajectory?** ###

#### **Introduction**

A golf ball acting as a moving projectile under gravity, while appearing to be a simple system, is in reality under the influence of multiple complex effects. Our project aims to determine the extent of how the trajectory a golf ball deviates due to its dimples and the velocity/spin-dependent Magnus effect. We aim to do this in three steps. First, we perform a momentum simulation of a specific golf ball (to reduce complexity) using a velocity inflow method. Then, using the results of such, we create a dataset to approximate and fit a function for the $S(v)$ parameter in the Magnus effect. Finally, we integrate the equations of motion of the macroscopic system to calculate and visualize trajectories. We compare the difference between an idealised model of a projectile and our model (gravity only and gravity + newtonian drag). The accuracy of our refined model will be tested using real-life data of golf-ball trajectories.

First, we shall import the necessary libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

#### **What is the Magnus Effect?**

The Magnus Effect is a physical phenomenon where a spinning object (golf ball) moving through a fluid medium (air) curves away from the expected path. This effect is very prominent in various sports like soccer, tennis, and in our project, golf. The Magnus effect is what causes the balls to curve in different directions depending on the spin. This is caused by the spin of the ball creating high and low pressure zones on each side of the ball, creating an induced force perpendicular to the path of motion. This is evident in the sport of golf when the ball is given topspin creating an induced force towards the ground sending the ball farther forward, or backspin creating an induced force towards the sky sending the ball higher in the air along its path.


\
\
<img src="magnus.png" width="500">

Importantly, the equation that governs this effect is 
$$\vec{F}_M = S(v)(\vec{v}\times\vec{\omega}),$$
which, when decomposed into co-ordinate forces, is
$$F_{Mx} = S(v) (\omega_zv_y - \omega_yv_z)$$
$$F_{My} = S(v) (\omega_zv_x - \omega_xv_z)$$
$$F_{Mz} = S(v) (\omega_yv_x - \omega_xv_y)$$
Where $S(v)$ is a parameter, that depends on velocity.



#### **Basic Projectile Under Gravity and Drag** ####

A simplified situation of a ball of under gravity can be compared to case with newtonian drag. The ball initially starts with an initial velocity $v_0$ at an angle $\theta$. Our coordinate system is positive upwards in y and positive to the right in x.

Other assumptions include a uniform density of air, a smooth ball and no spin of the ball. The diameter of the ball is taken to be 1.680 inches which is the minimum diameter allowed according to USGA Rules of Golf. Gravity only acts downwards thus there is an acceleration in the y-component of velocity. We can then write the equations of motion in x, y and the velocity components,

$$
\large
\frac{dx}{dt} = v_x
$$

$$
\large
\frac{dy}{dt} = v_y
$$

$$
\large
\frac{dv_x}{dt} = 0
$$

$$
\large
\frac{dv_y}{dt} = -g
$$

Accounting for newtonian drag, a extra term of force is added,

$$
\large
\vec{F}_D
=
-\frac{1}{2}\,\rho\,C_d\,A\,v\,\mathbf{v}
$$

$$
\large
\mathbf{v} = v_x\,\hat{x} + v_y\,\hat{y}, 
\quad
v = \|\vec{v}\| = \sqrt{v_x^2 + v_y^2}, 
\quad
$$
Where $\rho$ is the air density, A is the cross-sectional area and $C_d$ is the drag coefficient.
The drag force is a vector equation affecting both x and y components of velocity, modifying our equations of motion into,

$$
\large
\frac{dx}{dt} = v_x
$$

$$
\large
\frac{dy}{dt} = v_y
$$

$$
\large
\frac{dv_x}{dt}
=
-\frac{1}{2m}\,\rho\,C_d\,A\,v\,v_x
$$

$$
\large
\frac{dv_y}{dt}
=
-g \;-\; \frac{1}{2m}\,\rho\,C_d\,A\,v\,v_y
$$

$$
\large
v = \sqrt{v_x^2 + v_y^2}
$$

We can then define our constants and numerically solve the equations of motion,

In [ ]:
v0 = 70 #Assume initial velocity of 70m/s
theta0 = np.deg2rad(15) #Assume launch angle of 15 degrees
g = 9.81 #m/s^2
rho_0 = 1.293 #kg/m3
Cd = 0.47  #Drag coefficient relevant for the case of a projectile such as a golf ball
r = 0.042672/2 #Golf ball radius in m from a diameter of 1.68 inches
A = 4*np.pi*r**2
m = 0.04592623 #Mass of golf ball taken to be the maximum allowed by the USGA Rules of Golf (1.62 ounces) converted to kg
t = np.linspace(0, 6, 1000)

def newt(state, t):
    x, y, v_x, v_y = state
    return [v_x, v_y, 0, -g]

state0 = [0, 0, v0*np.cos(theta0), v0*np.sin(theta0)]

sol_newt = odeint(newt, state0, t)

def newt_drag(state, t):
    x, y, v_x, v_y = state
    return [v_x, v_y, -1/(2*m)*rho_0*Cd*A*np.sqrt(v_x**2+v_y**2)*v_x, -g-1/(2*m)*rho_0*Cd*A*np.sqrt(v_x**2+v_y**2)*v_y]

sol_drag = odeint(newt_drag, state0, t)

plt.figure(figsize=(7,6))
plt.title('Trajectory of a Golf Ball')
plt.plot(sol_newt[:,0], sol_newt[:,1], color='b', label='No Drag')
plt.plot(sol_drag[:,0], sol_drag[:,1], color='red', label='Newtonian Drag')
plt.ylim(0,20)
plt.xlabel('x [m]')
plt.ylabel('y [m]')
plt.legend()
plt.grid()
plt.show()

#### Simulating a Golf Ball in Python (the hard part)

As mentioned in the intro, the way we actually found values to use for the decomposition of S(v) is by using a rigid-body simulation of air and a golf ball. Our code for this is in the titular `golfsimulator.py` file, which was derived from a [bouncing ball simulation](https://github.com/viblo/pymunk/blob/master/pymunk/examples/bouncing_balls.py) from the pymunk examples library. We show the simulation in action:

![Image](simcap_2.gif)

To run the file for yourself, read the aptly named, `README.txt` file in the same folder as this notebook.

The basis of the physicality of the simulation is that we approximate air particles as perfectly elastic, rigid bodies. Importantly, we do not simulate every single air molecule, as that would be outside the scope of this project (we were told to keep the scope reined in) and would just be unfeasible on anything other than a supercomputer. Thus, we approximate air as a large quanitity of rigid body particles of fixed radii (about 2.34 mm) with density equal to the density of air. We believe this is a reasonable approximation, as long as the air particles are small enough to fit within the dimples of the golf ball, as that is where the "dragging" of air particles along its surface occurs, as can be seen in the gif above if one looks close enough.


We created the golf ball in the simulation using the `pymunk.autogeometry.march_soft()` function on a scaled down image of the curve of the golf ball created in Desmos, shown below.

![Image](GolfBallCurve.PNG)

![Image](GolfBallSmall.png)

Isn't the little one so cute?

Anyways, that creates a bunch of line segments, and we connect each one to a pymunk kinematic body, which is just the class that pymunk allows to do physics on other things but not recieve physics, i.e. all energies and position of the kinematic body should be the same from the beginning of the simulation to the end. The code for that is in lines 268-293 of the python file, under `def _add_good_golf_ball(self):`

I should mention that the air particles are practically identical save a few parameters like mass, radius, and elasticity, to the balls of the original pymunk example sim. 

Having created the objects we want to hit against each other, we needed to put them in the simulation, which for the golf ball, is easy, as we just call the creation function at init. However, for the air particles, this is not so simple. We generate them each frame, by calling their generation each frame, $n$ many times, where $n$ is the `--num2` parameter. This has the added side *benefit?* of over a long time, creating infinitely many particles, and computers, for now, are woefully finite, so we also add a function that gets called every frame that iterates over the list of particles and culls any that are outside the bounding box of the pipe and its open ends.

Now, we speak about the pipe. The pipe is made of two rectangles attached to static bodies, that have elasticity of near 1, as static bodies with elasticities of exactly one have some wierd behavior in pymunk. This was addressed in the `pygame-ce` fork, but the simulation had already been completed by the time we found out about the `pygame-ce` fork. Pretty simple on this front, but very useful, and gives rise to the 'pipe-condition'.

The 'pipe-condition' is the term we have been using for the idea that a the particles of a fluid moving in a pipe have an inverse quadratic relationship with distance from the center of the pipe, such that at the pipe edge, the velocity is classically, zero. We discussed this with you, Danny, and you made this statement, and we kinda rolled with it, so thats a pretty big approximation, that if faulty, nearly completely throws out the data we gathered from the simulation here. For this though, we initialized each of the particles along a single line, $x=10$, with a random $y$ position, and a velocity following a parabolic curve meeting the conditions of the center having $v_max$ velocity and touching the pipe having $0$ velocity, in the positive x-direction (which is towards the right for the simulation, incidentally, positive y is down in the simulation).

For the velocity, there is a litte extra that makes the simulation actually feasible, a scaling multiple of the velocity that, using dimensional analysis, converts the m/s values into pixels/frame, which is what the internal values use. We based this scaling off of the radius of the golf ball being 50 units, and since golf balls have a very specific radius, each unit must then be equal to 2.34mm. For the seconds to frame conversion, it gets a little more technical. We use the `

That aside, we then moved forward with actually simulating the physics of the situation, which is something that pymunk does natively! It is a physics simulation library, after all. But we want something very specific from our simulation, that of the force normal to the direction of airflow on the ball. To this end, thankfully, pymunk has our backs.

During our data collection, we ended up running the simulation over 500+ times. We simulated ball velocities from 70 meters per second all the way down to 1 meter per second. For each velocity, we ran 3 simulations with no rotation and 6 simulations with 1.0 rad/s rotation. Our process for this simulation was to split it up between us in the group. Running the simulation was very tedious and a single simulation would take upwards to 10 minutes to regulate so data could be taken. Having 3 separate devices was very useful and essentially cut our simulation time by 1/3. We also separated our simulations into separate public branches through VScode to make the final dataframes creation easier.

Through the simulation we collected our parallel force, normal force, and time of completion. The normal force is the value we must use to calculate the S(v) parameter.

#### **Obtaining s(v) Parameter** ####

When calculating our S(v) parameter, we used the interp1d function from the scipy.interpolate library to interpolate our normal force values into our S(v) parameter. In doing so, several debugging and data wrangling steps were needed to be taken before we processed our parameter. First, we had to make sure our dataframe was all numeric and >0 to avoid any division by zero errors. Next, we had to add a buffer to our dataframe since the interp1d function did not work well with repeating values. Finally, we had to set the interp1d kind to "cubic" to provide the most accurate data possible.

In [ ]:
import pandas as pd

golf_sim = pd.read_csv("force_output.csv") #Read Pandas Dataframe
golf_sim.columns = golf_sim.columns.str.strip().str.replace("'", "") #Make sure columns are readable by code

golf_sim["Angular velocity (rad/s)"] = pd.to_numeric(golf_sim["Angular velocity (rad/s)"], errors="coerce") #Fix Datatype issue

golf_rot = golf_sim[golf_sim["Angular velocity (rad/s)"] > 0] #Only view rotating values (division by 0 breaks simulation)
golf_rot = golf_rot.reset_index(drop=True) # re-index after culling
v = golf_rot['Velocity (m/s)']
Fn = golf_rot['Force normal v (N)'] #Establish v, Fn, and omega values for function
omega = golf_rot['Angular velocity (rad/s)']

In [ ]:
# separate for generating averaged values
buffer = []
avg_vals = []
for i in range(1,len(v)):
    if v[i] == v[i-1]:
        buffer.append((v[i-1],Fn[i-1],omega[i-1]))
        #print(buffer) debugging
        try:
            if v[i] != v[i+1]:
                buffer.append((v[i],Fn[i],omega[i]))
                buffer = np.array(buffer)
                v_avg = np.average(buffer[:,0])
                #print(buffer) debugging
                Fn_avg = np.average(buffer[:,1])
                omega_avg = np.average(buffer[:,2])
                avg_vals.append((v_avg,Fn_avg,omega_avg))
                buffer = []
        except:
            buffer.append((v[i],Fn[i],omega[i]))
            buffer = np.array(buffer)
            v_avg = np.average(buffer[:,0])
            Fn_avg = np.average(buffer[:,1])
            omega_avg = np.average(buffer[:,2])
            avg_vals.append((v_avg,Fn_avg,omega_avg))
            buffer = []
avg_vals = np.array(avg_vals)


In [ ]:
from scipy.interpolate import interp1d

v = avg_vals[:,0] 
Fn = avg_vals[:,1]
omega = avg_vals[:,2]

s = 2.34 * 4 * np.abs( Fn / ( v * omega ) ) #calculate s for each point. Small adjustment based on our simulation parameters.
s_of_v = interp1d(v, s, kind="cubic", fill_value="extrapolate") #Use interpolation function to solve for s(v) parameter


print(s_of_v(20)) #s(v) at 20 m/s
interp_v = np.linspace(0,70,1000)
plt.plot(interp_v,s_of_v(interp_v))
plt.grid()
plt.show()

As can be seen, this is incredibly nonlinear, with extremely non-trivial behavior. This is sort of what we expect though, as recorded in other experimental works on the magnus effect, specifically for golf balls, the $S(v)$ parameter is very much in flux, between some values.

With the S(v) parameter given a mostly continuous function, we can proceed to the integration and visualization.

#### Integrating EoM w/ Magnus Effect

We look at the equations of motion as before, now adding the contribution from the magnus effect to each axis:

$$\ddot{x} = -\frac{1}{2m}C_d\rho A\dot{x}\sqrt{\dot{x}^2 + \dot{y}^2 + \dot{z}^2} + \frac{S(|v|)}{m}(\omega_y\dot{z}-\omega_z\dot{y})$$

$$\ddot{y} = -\frac{1}{2m}C_d\rho A\dot{y}\sqrt{\dot{x}^2+\dot{y}^2+\dot{z}^2} + \frac{S(|v|)}{m}(\omega_z\dot{x} - \omega_x\dot{z})$$

$$\ddot{z} = -g-\frac{1}{2m}C_d\rho A\dot{z}\sqrt{\dot{x}^2+\dot{y}^2+\dot{z}^2} + \frac{S(|v|)}{m}(\omega_y\dot{x} - \omega_x\dot{y})$$

Exchanging these with their first-order counterparts in terms of $\dot{\vec{v}}$,

$$\dot{v}_x = -\frac{1}{2m}C_d\rho Av_x\sqrt{v_x^2 + v_y^2 + v_z^2} + \frac{S(|v|)}{m}(\omega_zv_y-\omega_yv_z)$$

$$\dot{v}_y = -\frac{1}{2m}C_d\rho Av_y\sqrt{v_x^2+v_y^2+v_z^2} + \frac{S(|v|)}{m}(\omega_zv_x - \omega_xv_z)$$

$$\dot{v}_z = -g-\frac{1}{2m}C_d\rho Av_z\sqrt{v_x^2+v_y^2+v_z^2} + \frac{S(|v|)}{m}(\omega_xv_y - \omega_yv_x)$$

$$\dot{x} = v_x$$
$$\dot{y} = v_y$$
$$\dot{z} = v_z$$

We must ask ourselves and the universe, exactly what should our initial values of $\omega$ be? We realize that the ball loses spin over time, at a consistent rate (which we have gleaned from some pro-golf sources). So, in addition to whatever we choose for the initial angular velocity, we implement a time-dependent decay of $\omega$. We made the same decision with regards to differences in ambient air pressure and temperature.

As for the values of $\omega$, many of our pro-golf sources explained that the average initial values for a pro-golfer's drive are
$$\vec{v}_0 = <70\cos(\theta),\pm 1, 70\sin(\theta)> \text{m/s}$$
$$\vec{\omega}_0 = <\pm50, -80\pi, 0> \text{rad/s}$$
Where the $\pm$ values indicate the values for slices and hooks, and are zero, when not slicing or hooking. We will not be looking at slicing or hooking, due to time constraints, and difficulties with visualizing the trajectories we encountered, due to the nature of the magnus effect, so we approximate

Now, we simply write these into coded functions:

In [ ]:
v0 = 70 #Assume initial velocity of 70m/s
theta0 = np.deg2rad(12) #Assume launch angle of 12 degrees
g = 9.81 #m/s^2
rho_0 = 1.293 * 1000 #g/m3
Cd = 0.380  #Drag coefficient relevant for the case of a projectile such as a golf ball
r = 0.042672/2 #Golf ball radius in m from a diameter of 1.68 inches
A = 4*np.pi*r**2
m = 0.04592623 * 1000 #Mass of golf ball taken to be the maximum allowed by the USGA Rules of Golf (1.62 ounces) converted to g
omega0 = [0,-243,0] #Looking at specifically no slice.
t_i_f = (0,25)
t = np.linspace(*t_i_f, 1000)
# we've seen these before!

Before we fully integrate the EoM for magnus effect, however, we would like to once more, visualize the non-magnus effect EoMs in 3-D instead of simply 2D. We also update it to account for the ball hitting the ground, and use the more up-to-date `solve_ivp()` instead of `odeint()`. The way we allow the ball to hit the ground here is by only terminating the solver after at least 0.2 seconds have passed, as the ball most certainly will not hit the ground at this time physically.

In [ ]:
from scipy.integrate import solve_ivp

def magnitude(v1,v2,v3):
    epsilon = 1e-8 #required so the derivative doesnt approach infinity as v approaches 0, as it is 1/(2sqrt(v))
    return np.sqrt(v1**2+v2**2+v3**2+epsilon)

def newt_3d(t,state):
    x, y, z, v_x, v_y, v_z = state #adapt it into position 3d, velocity 3d
    a_x = 0
    a_y = 0
    a_z = -g
    return [v_x, v_y, v_z, a_x, a_y, a_z] # 3d velocity, 3d acc.

def newt_drag_3d(t,state):
    x,y,z, v_x,v_y,v_z = state
    a_x = -(1/(2*m))*Cd*rho_0*A*v_x*magnitude(v_x,v_y,v_z)
    a_y = -(1/(2*m))*Cd*rho_0*A*v_y*magnitude(v_x,v_y,v_z)
    a_z = -g - (1/(2*m))*Cd*rho_0*A*v_z*magnitude(v_x,v_y,v_z)
    return [v_x, v_y, v_z, a_x, a_y, a_z]

def ground_ball(t,state):
    x,y,z, v_x,v_y,v_z = state
    if t < 0.2:
        return 1.0
    else:
        return z
ground_ball.terminal = True
ground_ball.direction= -1

state0 = [0, 0, 0.01, v0*np.cos(theta0), 0, v0*np.sin(theta0)] #3d init pos, 3d init vel
sol_newt_3d = solve_ivp(newt_3d, t_i_f, state0, t_eval=t, events=ground_ball) #solve it!
sol_drag_3d = solve_ivp(newt_drag_3d, t_i_f, state0, t_eval=t, events=ground_ball) #solve it again!

fig = plt.figure(figsize=(7,6))
ax = fig.add_subplot(111,projection="3d")
ax.plot(sol_newt_3d.y[0], sol_newt_3d.y[1], sol_newt_3d.y[2], label = "No Drag")
ax.plot(sol_drag_3d.y[0], sol_drag_3d.y[1], sol_drag_3d.y[2], label = "Drag")
ax.set_zlim((0,20))
ax.legend()
plt.show()


Exactly what we expect. Now to add the Magnus effect to it!

In [ ]:
def spinloss(t):
    return omega0[1] * (1 - 0.04)**t

def mag_3d(t,state):
    x,y,z ,v_x,v_y,v_z = state
    a_x = -(1/(2*m))*Cd*rho_0*A*v_x*magnitude(v_x,v_y,v_z) - (1/(m)) * 64 * s_of_v(v_x) * spinloss(t)*v_z #s_of_v(magnitude(v_x,v_y,v_z))
    a_y = -(1/(2*m))*Cd*rho_0*A*v_y*magnitude(v_x,v_y,v_z)
    a_z = -g - (1/(2*m))*Cd*rho_0*A*v_z*magnitude(v_x,v_y,v_z) - (1/(m)) * 4.5 * s_of_v(v_z) * spinloss(t)*v_x #small adjustments from some forums discussing this topic.
    return [v_x, v_y, v_z, a_x, a_y, a_z]

sol_mag_3d = solve_ivp(mag_3d, t_i_f, y0 = state0, t_eval = t, events = ground_ball, method="Radau") #use a solver more tuned for discrete problems (low predictability)


In [ ]:
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(111,projection="3d")
ax.plot(sol_newt_3d.y[0], sol_newt_3d.y[1], sol_newt_3d.y[2], label = "No Drag")
ax.plot(sol_drag_3d.y[0], sol_drag_3d.y[1], sol_drag_3d.y[2], label = "Drag")
ax.plot(sol_mag_3d.y[0], sol_mag_3d.y[1], sol_mag_3d.y[2], label = "Magnus Effect & Drag")
ax.set_zlim((0,35))
ax.set_xlabel("X-positon")
ax.set_ylabel("Y-position")
ax.set_zlabel("Z-position")
ax.legend()
fig.tight_layout(h_pad=1)
plt.show()

And this is what we get. 
We can see the drag and no drag as usual, though they look smaller due to the increased bounding box. We see that the magnus effect and drag position has a large lift, following almost linearly from the other cases, and with a sharp, but rounded decrease as the ball reaches its apex, and falls. This is exactly what we expect. Sparing the code, the maximum height of this is about 31m, which is exactly the maximum height of a real driver swing with these parameters. 

Importantly, to us - actually, the most important thing here - is that the range of the magnus trajectory is noticeably greater than the range of the parabolic case, as well as significantly greater than the range of the drag trajectory. This is exactly what happens in true golf drives. With an incident angle of 12 degrees, a range of 250m = 273y is very reasonable.

We now compare this to our trajectory from the midterm 1 exercise 1. 

![Image](Plot1Midterm1.png)

Obviously, we have a slightly different launch angle, but the quantitative difference is negligible when viewing the qualitative results here.
The drag and gravity cases seem similar to our more recent analysis, which is a good thing, although in this case it seems that the range on both is sizably greater. The magnus effect trajectories however, are not so similar. In the old analysis, the magnus effect trajectory was just barely better in range and height than the drag trajectory. In our updated trajectories though, the magnus effect trajectory is significantly greater in range and height, as mentioned before, than both the drag-only and non-drag cases.

This affirms to us that we have done at least a decent job at modeling this trajectory, and that our efforts were not in vain.

What remains to be done (not in this class or essay) would be to formulate an experiment wherein a golf ball is struck in conditions to create the same backspin and initial velocity as we have here, and utilize some sort of imaging device to create a digital representation of the trajectory of the ball (as well as the range and maximum heights, but those are encoded within the trajectory), so that we may compare that to our simulated data. This would offer the greatest information as to whether our predictions for $S(v)$ are correct, or if they are faulty.

There are some real datasets for trajectories online, but the ones we were able to find were relatively terrible for our purposes, as they didn't indicate the initial velocity, or the backspin on the ball, which made it unfeasible for us to use these as comparisons.

Thank you for the time allocated to reading our work, and for all the generous support you've given us over the course of this project, and the semester as a whole.